In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import numpy as np
import pandas as pd
import os
import pickle
import gc
# 分布確認
# import pandas_profiling as pdp
# 可視化
import matplotlib.pyplot as plt
# 分布確認
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
# モデリング
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
import lightgbm as lgb

import warnings
warnings.filterwarnings('ignore')

# !pip install japanize-matplotlib
# import japanize_matplotlib
%matplotlib inline

In [ ]:
df_train = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
df_train.head()

In [ ]:
print(df_train.shape)
print('レコード数：', len(df_train))
print('カラム数：', len(df_train.columns))

In [ ]:
df_train.info()

In [ ]:
df_train['Pclass'] = df_train['Pclass'].astype(object)
df_train[['Pclass']].info()

In [ ]:
df_train['Pclass'] = df_train['Pclass'].astype(np.int64)
df_train[['Pclass']].info()

In [ ]:
df_train.isnull().sum()

In [ ]:
x_train, y_train, id_train = df_train[['Pclass', 'Fare']], df_train[['Survived']], df_train[['PassengerId']]
print(x_train.shape, y_train.shape, id_train.shape)

In [ ]:
x_tr, x_va, y_tr, y_va = train_test_split(x_train, y_train, test_size=0.2, shuffle=True, stratify=y_train, random_state=123)
print(x_tr.shape, y_tr.shape)
print(x_va.shape, y_va.shape)
print('y_train:{:.3f}, y_tr:{:.3f}, y_va:{:.3f}'.format(y_train['Survived'].mean(), y_tr['Survived'].mean(), y_va['Survived'].mean(),))

In [ ]:
n_splits = 5
cv = list(StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=123).split(x_train, y_train))

for nfold in np.arange(n_splits):
    print('-'*20, nfold, '-'*20)
    idx_tr, idx_va = cv[nfold][0], cv[nfold][1]
    x_tr, y_tr = x_train.loc[idx_tr, :], y_train.loc[idx_tr, :]
    x_va, y_va = x_train.loc[idx_va, :], y_train.loc[idx_va, :]
    print(x_tr.shape, y_tr.shape)
    print(x_va.shape, y_va.shape)
    print('y_train:{:.3f}, y_tr:{:.3f}, v_va:{:.3f}'.format(
        y_train['Survived'].mean(),
        y_tr['Survived'].mean(),
        y_va['Survived'].mean(),
    ))

In [ ]:
params = {
    'boosting_type': 'gbdt',
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.1,
    'num_leaves': 16,
    'n_estimators': 100000,
    'random_state': 123,
    'importance_type': 'gain',
}

model = lgb.LGBMClassifier(**params)
model.fit(x_tr, y_tr, eval_set=[(x_tr, y_tr), (x_va, y_va)],)

In [ ]:
y_tr_pred = model.predict(x_tr)
y_va_pred = model.predict(x_va)
metric_tr = accuracy_score(y_tr, y_tr_pred)
metric_va = accuracy_score(y_va, y_va_pred)
print('[accuracy] tr: {:.2f}, va: {:.2f}'.format(metric_tr, metric_va))

In [ ]:
imp = pd.DataFrame({'col': x_train.columns, 'imp': model.feature_importances_})
imp.sort_values('imp', ascending=False, ignore_index=True)

In [ ]:
params = {
    'boosting_type': 'gbdt',
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.1,
    'num_leaves': 16,
    'n_estimators': 100000,
    'random_state': 123,
    'importance_type': 'gain',
}

metrics = []
imp = pd.DataFrame()

n_splits = 5
cv = list(StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=123).split(x_train, y_train))

for nfold in np.arange(n_splits):
    print('-'*20, nfold, '-'* 20)
    idx_tr, idx_va = cv[nfold][0], cv[nfold][1]
    x_tr, y_tr = x_train.loc[idx_tr, :], y_train.loc[idx_tr, :]
    x_va, y_va = x_train.loc[idx_va, :], y_train.loc[idx_va, :]
    print(x_tr.shape, y_tr.shape)
    print(x_va.shape, y_va.shape)
    print('y_train:{:.3f}, y_tr:{:.3f}, y_va:{:.3f}'.format(
        y_train['Survived'].mean(),
        y_tr['Survived'].mean(),
        y_va['Survived'].mean(),
    ))

    model = lgb.LGBMClassifier(**params)
    model.fit(x_tr, y_tr, eval_set=[(x_tr, y_tr), (x_va, y_va)],)

    y_tr_pred = model.predict(x_tr)
    y_va_pred = model.predict(x_va)
    metric_tr = accuracy_score(y_tr, y_tr_pred)
    metric_va = accuracy_score(y_va, y_va_pred)
    print('[accuracy] tr: {:.2f}, va: {:.2f}'.format(metric_tr, metric_va))
    metrics.append([nfold, metric_tr, metric_va])

    _imp = pd.DataFrame({'col': x_train.columns, 'imp': model.feature_importances_, 'nfold': nfold})
    imp = pd.concat([imp, _imp], axis=0, ignore_index=True)

print('-'*20, 'result', '-'*20)
metrics = np.array(metrics)
print(metrics)

print('[cv ] tr: {:.2f}+-{:.2f}, va: {:.2f}+-{:.2f}'.format(
    metrics[:, 1].mean(), metrics[:, 1].std(),
    metrics[:, 2].mean(), metrics[:, 2].std(),
))

imp = imp.groupby('col')['imp'].agg(['mean', 'std'])
imp.columns = ['imp', 'imp.std']
imp = imp.reset_index(drop=False)

print('Done.')

In [ ]:
x_tr, x_va2, y_tr, y_va2 = train_test_split(x_train, y_train, test_size=0.2, shuffle=True, stratify=y_train, random_state=123)
print(x_tr.shape, y_tr.shape)
print(x_va2.shape, y_va2.shape)

In [ ]:
x_tr1, x_va1, y_tr1, y_va1 = train_test_split(x_tr, y_tr, test_size=0.2, shuffle=True, stratify=y_tr, random_state=789)
print(x_tr1.shape, y_tr1.shape)
print(x_va1.shape, y_va1.shape)

In [ ]:
params = {
    'boosting_type': 'gbdt',
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.1,
    'num_leaves': 16,
    'n_estimators': 100000,
    'random_state': 123,
    'importance_type': 'gain',
}

model = lgb.LGBMClassifier(**params)
model.fit(x_tr1, y_tr1, eval_set=[(x_tr1, y_tr1), (x_va1, y_va1)],)

In [ ]:
y_va1_pred = model.predict(x_va1)
y_va2_pred = model.predict(x_va2)

In [ ]:
print('[検証データ acc: {:.4f}'.format(accuracy_score(y_va1, y_va1_pred)))
print('[ベースライン検証データ] acc: {:.4f}'.format(accuracy_score(y_va2, y_va2_pred)))

In [ ]:
print('検証データ')
print(confusion_matrix(y_va1, y_va1_pred))
print(confusion_matrix(y_va1, y_va1_pred, normalize='all'))
print('ベースライン検証データ')
print(confusion_matrix(y_va2, y_va2_pred))
print(confusion_matrix(y_va2, y_va2_pred, normalize='all'))

In [ ]:
y_va1_pred_prob = model.predict_proba(x_va1)[:,1]
y_va2_pred_prob = model.predict_proba(x_va2)[:,1]

fig = plt.figure(figsize=(10,8))

fig.add_subplot(2, 1, 1)
plt.title('検証データ1')
plt.hist(y_va1_pred_prob[np.array(y_va1).reshape(-1)==1], bins=10, alpha=0.5, label='1')
plt.hist(y_va1_pred_prob[np.array(y_va1).reshape(-1)==0], bins=10, alpha=0.5, label='0')

fig.add_subplot(2, 1, 2)
plt.title('ベースライン検証データ2')
plt.hist(y_va2_pred_prob[np.array(y_va2).reshape(-1)==1], bins=10, alpha=0.5, label='1')
plt.hist(y_va2_pred_prob[np.array(y_va2).reshape(-1)==0], bins=10, alpha=0.5, label='0')

plt.grid()
plt.legend()

In [ ]:
df_test = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')
x_test = df_test[['Pclass', 'Fare']]
id_test = df_test[['PassengerId']]

In [ ]:
y_test_pred = model.predict(x_test)

In [ ]:
df_submit = pd.DataFrame({'PassengerId':id_test['PassengerId'], 'Survived':y_test_pred})
display(df_submit.head(5))

In [ ]:
df_submit.to_csv('submission_baseline.csv', index=None)